<a href="https://colab.research.google.com/github/Daria-Lytvynenko/ML_course/blob/main/bank_deposit_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ПРОГНОЗУВАННЯ ОФОРМЛЕННЯ ДЕПОЗИТІВ КЛІЄНТАМИ БАНКУ

# 0. ІМПОРТИ ТА КОНФІГ

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
from plotly import graph_objects as go

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (confusion_matrix, roc_auc_score, roc_curve,
                             f1_score, auc, classification_report)
from sklearn.preprocessing import (OneHotEncoder, OrdinalEncoder,
                                   StandardScaler)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree

from imblearn.over_sampling import SMOTENC

import lightgbm as lgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

from google.colab import drive

In [ ]:
#Константи
RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.15
DECISION_THRESHOLD = 0.6

1. ЗАВАНТАЖЕННЯ ДАНИХ

In [ ]:
drive.mount('/content/drive')

train = pd.read_csv('drive/MyDrive/ML_course/bank-additional-full.csv', sep=";")

print(train.head())
print(train.info())

# Видаляємо duration (leakage — відомо лише після дзвінка)
train.drop('duration', axis=1, inplace=True)

# 2. EDA

In [ ]:
#Розподіл цільової змінної
print(train['y'].value_counts(normalize=True))

train['y'] = np.where(train['y'] == 'yes', 1, 0)

X = train.iloc[:, :-1].copy()
y = train['y']

#Кореляційна матриця числових ознак
sns.heatmap(train.select_dtypes(exclude='object').corr())
plt.tight_layout()
plt.show()

Класи не збалансовані: позитивний клас ~11.2%, негативний ~88.8%.
Для балансування використовуємо SMOTENC — підходить для змішаних даних
(категоріальні + числові). Економічні показники залишаємо всі,
оскільки стан економіки напряму впливає на платоспроможність клієнтів.




In [ ]:
#Гістограми розподілу цільової змінної
def distribution_plots(dataset, idx):
    nrows = len(idx)
    fig, axes = plt.subplots(nrows=nrows, ncols=1, figsize=(6, 4 * nrows))
    if nrows == 1:
        axes = [axes]

    for i, col in enumerate(idx):
        sns.histplot(
            data=dataset,
            x=col,
            hue='y',
            bins=20,
            multiple='fill',
            ax=axes[i],
            kde=True,
            palette='vlag'
        )
        axes[i].set_title(f'Розподіл цільової змінної для {col}', fontsize=12)
        axes[i].set_ylabel('Частка (0 до 1)')
        axes[i].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

In [ ]:
# Соціально-економічні атрибути
social_idx = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
              'euribor3m', 'nr.employed']
distribution_plots(train, social_idx)

In [ ]:
# Персональні дані
personal_idx = ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']
distribution_plots(train, personal_idx)

In [ ]:
# Останній контакт
contact_idx = ['contact', 'month', 'day_of_week']
distribution_plots(train, contact_idx)

In [ ]:
# Інші параметри
other_idx = ['campaign', 'previous', 'poutcome']
distribution_plots(train, other_idx)

In [ ]:
# pdays=999 означає відсутність попереднього контакту — виводимо окремо
pdays_df = train[train['pdays'] < 999][['pdays', 'y']]
plt.figure(figsize=(6, 4))
sns.histplot(data=pdays_df, x='pdays', hue='y', bins=20,
             multiple='fill', kde=True, palette='vlag')
plt.title('Розподіл цільової змінної для pdays', fontsize=12)
plt.ylabel('Частка (0 до 1)')
plt.xlabel('Кількість днів після останнього контакту')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# 3. ОБГРУНТУВАННЯ МЕТРИКИ

Використовуємо F1-метрику та орієнтуємось на зменшення хибнонегативних
результатів: пропустити потенційного клієнта дорожче, ніж витратити
зусилля на того, хто відмовиться. Хибнопозитивні прогнози збільшують
маркетингові витрати, але є менш критичними.



# 4. ПРЕПРОЦЕСИНГ


##4.1 Виявлення викидів


In [ ]:
num_cols = train.select_dtypes(exclude='object').columns.tolist()
num_cols.remove('y')

def outliers(columns, dataset):
    outlier_list = []
    for col in columns:
        data = dataset[dataset[col] < 999][col] if col == 'pdays' else dataset[col]
        q1 = np.percentile(data, 25)
        q3 = np.percentile(data, 75)
        iqr = q3 - q1
        lower_value = q1 - 1.5 * iqr
        upper_value = q3 + 1.5 * iqr
        outlier_list.append({
            'col': col,
            'lower_bound': round(lower_value, 2),
            'upper_bound': round(upper_value, 2),
            'count_below': len(data[data < lower_value]),
            'count_above': len(data[data > upper_value])
        })
    return pd.DataFrame(outlier_list)

outlier_stats = outliers(num_cols, train)
print(outlier_stats)


In [ ]:
# Кліпінг викидів до верхньої межі IQR: pdays: 13, campaign: 6, age: 69
for _, row in outlier_stats.iterrows():
    col = row['col']
    if col == 'pdays':
        mask = train[col] < 999
        train.loc[mask, col] = train.loc[mask, col].clip(upper=row['upper_bound'])
    else:
        train[col] = train[col].clip(upper=row['upper_bound'])

##4.2 Графіки викидів

In [ ]:
def outlier_plots(dataset, out_cols):
    fig, axes = plt.subplots(nrows=len(out_cols), ncols=1,
                             figsize=(6, 4 * len(out_cols)))
    if len(out_cols) == 1:
        axes = [axes]

    for i, col in enumerate(out_cols):
        sns.scatterplot(data=dataset, x=col, y='y',
                        hue='y', ax=axes[i], alpha=0.6, palette='coolwarm')
        q1 = dataset[col].quantile(0.25)
        q3 = dataset[col].quantile(0.75)
        upper_limit = q3 + 1.5 * (q3 - q1)
        axes[i].axvline(upper_limit, color='blue', linestyle='--',
                        label=f'Межа IQR ({upper_limit:.1f})')
        axes[i].legend(fontsize=8)
        axes[i].set_title(f'Викиди для {col}', fontsize=12)
        axes[i].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

out_cols = ['age', 'campaign', 'pdays', 'previous', 'cons.conf.idx']
outlier_plots(train, out_cols)

##4.3 Додаткові розподіли

In [ ]:
for col in ['age', 'campaign', 'previous']:
    plt.hist(train[col], bins=20)
    plt.title(col)
    plt.show()

plt.hist(train[train['pdays'] <= 18]['pdays'], bins=6)
plt.title('pdays (без sentinel 999)')
plt.show()

for col in ['campaign', 'previous', 'age']:
    sns.histplot(data=train, x=col, hue='y', multiple='fill',
                 shrink=0.8, stat='proportion')
    plt.ylabel('Частка (від 0 до 1)')
    plt.title(f'Розподіл за {col}')
    plt.show()

print(train.describe())

#5. FEATURE ENGINEERING

In [ ]:
X = train.iloc[:, :-1].copy()
y = train['y']

# Наявність попереднього контакту
X['prev_contact'] = np.where(X['pdays'] == 999, 'yes', 'no')

# Маркер залученості до попередньої кампанії
X['prev_campaign'] = np.where(
    (X['poutcome'] == 'failure') | (X['poutcome'] == 'success'), 'yes', 'no'
)

# Неуспішна попередня кампанія + відсутність контакту
X['prev_fail_no_contact'] = np.where(
    (X['poutcome'] == 'failure') & (X['pdays'] == 999), 'yes', 'no'
)


# 6. TRAIN / VALIDATION / TEST SPLIT

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

val_relative_size = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=val_relative_size,
    random_state=RANDOM_STATE,
    stratify=y_trainval
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

#7. PREPROCESSING


In [ ]:
numeric_cols = X.select_dtypes(exclude='object').columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

ordinal_col = ['education']
for col in ordinal_col:
    categorical_cols.remove(col)

education_order = ['unknown', 'illiterate', 'basic.4y', 'high.school',
                   'basic.6y', 'basic.9y', 'professional.course',
                   'university.degree']

numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
one_hot_enc = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])
ordinal_enc = Pipeline(steps=[('ordinal', OrdinalEncoder(
    categories=[education_order], handle_unknown='use_encoded_value',
    unknown_value=-1))])

preprocessor = ColumnTransformer(transformers=[
    ('onehot',   one_hot_enc,         categorical_cols),
    ('ordinal',  ordinal_enc,         ordinal_col),
    ('scaler',   numeric_transformer, numeric_cols)
], remainder='passthrough')

#8. ДОПОМІЖНІ ФУНКЦІЇ

In [ ]:
def evaluate(model, X_tr, X_val, y_tr, y_val, label=''):
    p_tr  = model.predict_proba(X_tr)[:, 1]
    p_val = model.predict_proba(X_val)[:, 1]
    auc_tr  = roc_auc_score(y_tr, p_tr)
    auc_val = roc_auc_score(y_val, p_val)
    pred_val = model.predict(X_val)
    f1_val = f1_score(y_val, pred_val)
    print(f"\n{'='*40}\n{label}")
    print(f"ROC-AUC  train: {auc_tr:.4f} | val: {auc_val:.4f}")
    print(f"F1       val:   {f1_val:.4f}")
    print(classification_report(y_val, pred_val, digits=4))
    return {'label': label, 'auc_train': auc_tr, 'auc_val': auc_val, 'f1_val': f1_val}

def best_threshold(model, X_val, y_val):
    proba = model.predict_proba(X_val)[:, 1]
    _, _, thresholds = roc_curve(y_val, proba)
    best_f1, best_tr = 0, 0.5
    for tr in thresholds:
        pred = (proba >= tr).astype(int)
        f = f1_score(y_val, pred)
        if f > best_f1:
            best_f1, best_tr = f, tr
    print(f"Найкращий поріг на val: {best_tr:.3f} → F1={best_f1:.4f}")
    return best_tr

results = []

#9. Models

##Logistic Regression

In [ ]:
logreg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        solver='liblinear', penalty='l1',
        class_weight={1: 8, 0: 1},
        random_state=RANDOM_STATE
    ))
])

logreg_pipeline.fit(X_train, y_train)
results.append(evaluate(logreg_pipeline, X_train, X_val, y_train, y_val,
                         label='Logistic Regression'))

# Підбір порогу на валідаційному наборі
lr_threshold = best_threshold(logreg_pipeline, X_val, y_val)

# Оцінка на тесті з підібраним порогом
proba_test_lr = logreg_pipeline.predict_proba(X_test)[:, 1]
pred_test_lr  = (proba_test_lr >= lr_threshold).astype(int)
print("\n--- LogReg на TEST ---")
print(classification_report(y_test, pred_test_lr, digits=4))

# Feature importance
feat_names = logreg_pipeline.named_steps['preprocessor'].get_feature_names_out()
coefs      = logreg_pipeline.named_steps['classifier'].coef_[0]
feat_imp_lr = (pd.DataFrame({'feature': feat_names, 'coef': coefs})
               .sort_values('coef', ascending=False))
print(feat_imp_lr.head(20).to_string())

# Таблиця помилок
test_pred_logreg = pd.concat([
    X_test, y_test,
    pd.Series(pred_test_lr, index=X_test.index, name='pred')
], axis=1)
print(f"\nПомилкові передбачення: {(test_pred_logreg['y'] != test_pred_logreg['pred']).sum()}")

## SMOTENC


In [ ]:
X_train_proc = pd.DataFrame(
    preprocessor.fit_transform(X_train),
    columns=preprocessor.get_feature_names_out()
)
X_val_proc  = pd.DataFrame(
    preprocessor.transform(X_val),
    columns=preprocessor.get_feature_names_out()
)
X_test_proc = pd.DataFrame(
    preprocessor.transform(X_test),
    columns=preprocessor.get_feature_names_out()
)

feature_names = preprocessor.get_feature_names_out().tolist()
cat_mask = [name.startswith('onehot__') or name.startswith('ordinal__')
            for name in feature_names]
cat_indices = [i for i, m in enumerate(cat_mask) if m]

smotenc = SMOTENC(random_state=RANDOM_STATE, categorical_features=cat_indices)
X_train_sm, y_train_sm = smotenc.fit_resample(X_train_proc, y_train)
print(f"Після SMOTENC: {X_train_sm.shape}, розподіл y: {pd.Series(y_train_sm).value_counts().to_dict()}")

## Logistic Regression with SMOTENC


In [ ]:
logreg_sm = LogisticRegression(
    solver='saga', penalty='l2', random_state=RANDOM_STATE, max_iter=1000
)
logreg_sm.fit(X_train_sm, y_train_sm)
results.append(evaluate(logreg_sm, X_train_sm, X_val_proc, y_train_sm, y_val,
                         label='Logistic Regression + SMOTENC'))

lr_sm_threshold = best_threshold(logreg_sm, X_val_proc, y_val)

proba_test_lr_sm = logreg_sm.predict_proba(X_test_proc)[:, 1]
pred_test_lr_sm  = (proba_test_lr_sm >= lr_sm_threshold).astype(int)
print("\n--- LogReg+SMOTENC на TEST ---")
print(classification_report(y_test, pred_test_lr_sm, digits=4))

# Аналіз розподілу ймовірностей
thresholds_df = pd.concat([
    y_test.reset_index(drop=True),
    pd.Series(logreg_sm.predict(X_test_proc), name='pred'),
    pd.Series(proba_test_lr_sm, name='pred_proba')
], axis=1)

for label, mask in [
    ('Всі',                    slice(None)),
    ('FP (0 передбачено як 1)', (thresholds_df['y'] == 0) & (thresholds_df['pred'] == 1)),
    ('FN (1 передбачено як 0)', (thresholds_df['y'] == 1) & (thresholds_df['pred'] == 0)),
    ('Справжні позитивні',      (thresholds_df['y'] == 1) & (thresholds_df['pred'] == 1)),
]:
    subset = thresholds_df[mask] if not isinstance(mask, slice) else thresholds_df
    plt.hist(subset['pred_proba'], bins=20)
    plt.title(f'Розподіл ймовірностей — {label}')
    plt.xlabel('pred_proba')
    plt.show()

# Feature importance
logreg_sm_imp = (pd.DataFrame({
    'features': feature_names,
    'importance': np.round(logreg_sm.coef_[0], 4)
}).sort_values('importance', ascending=False))
print(logreg_sm_imp.head(20).to_string())


## KNN

In [ ]:
def search_model(search, X_tr, y_tr):
    search.fit(X_tr, y_tr)
    print(f"Найкращі параметри: {search.best_params_}")
    print(f"Найкращий score:    {search.best_score_:.4f}")
    return search.best_estimator_

# Базовий KNN
knn_base = KNeighborsClassifier()
knn_base.fit(X_train_sm, y_train_sm)
results.append(evaluate(knn_base, X_train_sm, X_val_proc, y_train_sm, y_val,
                         label='KNN (base)'))

# Пошук гіперпараметрів
params_knn = {'n_neighbors': np.arange(1, 20)}
knn_best = search_model(
    GridSearchCV(KNeighborsClassifier(), param_grid=params_knn,
                 cv=5, scoring='f1'),
    X_train_sm, y_train_sm
)
knn_best.fit(X_train_sm, y_train_sm)
results.append(evaluate(knn_best, X_train_sm, X_val_proc, y_train_sm, y_val,
                         label='KNN (tuned)'))

## Decision Tree


In [ ]:
params_tree = {
    'max_depth':      np.arange(1, 20, 1),
    'max_leaf_nodes': np.arange(2, 10, 1)
}
dt_best = search_model(
    GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE),
                 param_grid=params_tree, cv=5, scoring='roc_auc'),
    X_train_sm, y_train_sm
)
dt_best.fit(X_train_sm, y_train_sm)
results.append(evaluate(dt_best, X_train_sm, X_val_proc, y_train_sm, y_val,
                         label='Decision Tree'))

# Feature importance
tree_feat_imp = (pd.DataFrame({
    'features':   feature_names,
    'importance': np.round(dt_best.feature_importances_, 4)
}).sort_values('importance', ascending=False))
print(tree_feat_imp.head(10).to_string())

plt.figure(figsize=(16, 12))
plot_tree(dt_best, feature_names=feature_names, filled=True)
plt.tight_layout()
plt.show()

##LightGBM


In [ ]:
X_train_lgb = X_train.copy()
X_val_lgb   = X_val.copy()
X_test_lgb  = X_test.copy()

cat_cols_lgb = X_train_lgb.select_dtypes(include='object').columns.tolist()
for col in cat_cols_lgb:
    X_train_lgb[col] = X_train_lgb[col].astype('category')
    X_val_lgb[col]   = X_val_lgb[col].astype('category')
    X_test_lgb[col]  = X_test_lgb[col].astype('category')

smotenc_lgb = SMOTENC(random_state=RANDOM_STATE, categorical_features=cat_cols_lgb)
X_tr_lgb_sm, y_tr_lgb_sm = smotenc_lgb.fit_resample(X_train_lgb, y_train)

for col in cat_cols_lgb:
    X_tr_lgb_sm[col] = X_tr_lgb_sm[col].astype('category')

###Hyperopt


In [ ]:
def objective(params):
    clf = lgb.LGBMClassifier(
        n_estimators=int(params['n_estimators']),
        learning_rate=params['learning_rate'],
        max_depth=int(params['max_depth']),
        num_leaves=int(params['num_leaves']),
        min_child_weight=params['min_child_weight'],
        subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
        reg_alpha=params['reg_alpha'],
        reg_lambda=params['reg_lambda'],
        min_split_gain=params['min_split_gain'],
        scale_pos_weight=params['scale_pos_weight'],
        random_state=RANDOM_STATE,
        verbose=-1
    )
    clf.fit(X_tr_lgb_sm, y_tr_lgb_sm,
            eval_set=[(X_val_lgb, y_val)],
            categorical_feature=cat_cols_lgb)
    proba = clf.predict_proba(X_val_lgb)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, proba)
    roc_auc = auc(fpr, tpr)
    return {'loss': -roc_auc, 'status': STATUS_OK}


space = {
    'n_estimators':    hp.quniform('n_estimators', 50, 500, 25),
    'learning_rate':   hp.uniform('learning_rate', 0.01, 0.3),
    'max_depth':       hp.quniform('max_depth', 3, 15, 1),
    'num_leaves':      hp.quniform('num_leaves', 20, 200, 1),
    'min_child_weight':hp.quniform('min_child_weight', 1, 10, 1),
    'subsample':       hp.uniform('subsample', 0.5, 1.0),
    'colsample_bytree':hp.uniform('colsample_bytree', 0.5, 1.0),
    'reg_alpha':       hp.uniform('reg_alpha', 0, 1),
    'reg_lambda':      hp.uniform('reg_lambda', 0, 1),
    'min_split_gain':  hp.uniform('min_split_gain', 0, 0.1),
    'scale_pos_weight':hp.uniform('scale_pos_weight', 0.1, 10)
}

trials = Trials()
best_params = fmin(fn=objective, space=space, algo=tpe.suggest,
                   max_evals=40, trials=trials, rstate=np.random.default_rng(RANDOM_STATE))

for p in ['n_estimators', 'max_depth', 'num_leaves', 'min_child_weight']:
    best_params[p] = int(best_params[p])

print("Найкращі гіперпараметри:", best_params)

###LGBM with best_params

In [ ]:
final_lgb_clf = lgb.LGBMClassifier(
    **best_params,
    random_state=RANDOM_STATE,
    verbose=-1
)
final_lgb_clf.fit(
    X_tr_lgb_sm, y_tr_lgb_sm,
    eval_set=[(X_val_lgb, y_val)],
    categorical_feature=cat_cols_lgb
)

lgb_threshold = best_threshold(final_lgb_clf, X_val_lgb, y_val)

proba_test_lgb = final_lgb_clf.predict_proba(X_test_lgb)[:, 1]
pred_test_lgb  = (proba_test_lgb >= lgb_threshold).astype(int)

fpr, tpr, _ = roc_curve(y_test, proba_test_lgb)
roc_auc_lgb = auc(fpr, tpr)
print(f"\nLGBM — ROC-AUC на test: {roc_auc_lgb:.4f}")
print(classification_report(y_test, pred_test_lgb, digits=4))

results.append({
    'label': 'LightGBM (hyperopt)',
    'auc_val': roc_auc_score(y_val, final_lgb_clf.predict_proba(X_val_lgb)[:, 1]),
    'auc_train': roc_auc_score(y_tr_lgb_sm, final_lgb_clf.predict_proba(X_tr_lgb_sm)[:, 1]),
    'f1_val': f1_score(y_val, final_lgb_clf.predict(X_val_lgb))
})

#10. ФІНАЛЬНЕ ПОРІВНЯННЯ МОДЕЛЕЙ

In [ ]:
results_df = pd.DataFrame(results).sort_values('f1_val', ascending=False)
print("\n" + "="*60)
print("ПОРІВНЯННЯ МОДЕЛЕЙ (за F1 на val set)")
print("="*60)
print(results_df[['label', 'auc_train', 'auc_val', 'f1_val']].to_string(index=False))

#ВІЗУАЛІЗАЦІЯ КАТЕГОРІАЛЬНИХ ЗМІННИХ (EDA)


In [ ]:
#Інтерактивні графіки розподілу категоріальних змінних за цільовою колонкою.
def custom_subplots(dataset, target_col='y'):
    cat_data = dataset.select_dtypes(include='category')
    cols = [c for c in cat_data.columns if c != target_col]
    n = len(cols)
    rows = (n + 1) // 2

    fig = make_subplots(rows=rows, cols=2, subplot_titles=cols)
    for idx, col in enumerate(cols, start=1):
        subset = (dataset.groupby([col])[[col, target_col]]
                  .value_counts(normalize=True)
                  .reset_index())
        for val in subset[target_col].unique():
            sub = subset[subset[target_col] == val]
            fig.add_trace(
                go.Bar(x=sub[col], y=sub['proportion'], name=str(val)),
                row=(idx + 1) // 2, col=(idx % 2) + 1
            )
    fig.update_layout(barmode='group', width=1200, height=400 * rows,
                      title_text=f'Розподіл за {target_col}')
    fig.show()


# Розподіл за реальним y
X_test_cat = X_test.copy()
for col in X_test_cat.select_dtypes(include='object').columns:
    X_test_cat[col] = X_test_cat[col].astype('category')

test_pred_full = pd.concat([
    X_test_cat, y_test.reset_index(drop=True),
    pd.Series(pred_test_lgb, name='pred')
], axis=1)

custom_subplots(test_pred_full, target_col='y')
custom_subplots(test_pred_full, target_col='pred')

# pdays розподіл (інтерактивний)
pdays_grp = (train[train['pdays'] < 999]
             .groupby('pdays')['y']
             .value_counts(normalize=True)
             .reset_index(name='proportion'))
# fig = px.histogram(pdays_grp, x='pdays', y='proportion', color='y',
#                    barmode='group', histfunc='avg',
#                    nbins=train[train['pdays'] < 999]['pdays'].nunique() * 2)
# fig.update_layout(width=900, height=450)
# fig.show()

#ДОПОМІЖНІЙ АНАЛІЗ (EDA — пропорції категоріальних змінних)

In [ ]:
def proportions(dataset):
    """
    Зводна таблиця частот для всіх категоріальних колонок (крім y).
    """
    frames = []
    for col in dataset.select_dtypes(include='object').columns:
        if col == 'y':
            continue
        counts = (dataset[col]
                  .value_counts(normalize=True)
                  .reset_index()
                  .rename(columns={'index': 'value', col: 'proportion'}))
        counts.insert(0, 'feature', col)
        frames.append(counts)
    return pd.concat(frames, ignore_index=True)

print(proportions(train).to_string())

# px.box(train['previous'], title='Розподіл previous').show()